%md
# CSV to Delta Ingestion Framework - Batch Process

Pattern: Control table + Log table -> the log table drives fault-tolerant retries.
Re-running this notebook automatically skips any file already marked success and
reprocesses everything else (if failed).

**Control table schema:**

| Column | Meaning |
|---|---|
| S_No | Serial Number |
| DataMart | Data Domain |
| Target_Catalog | Delta target catalog |
| Target_Schema | Delta target schema |
| Target_Table_Name | Delta target table |
| Source_Path | folder path (e.g. `s3://bucket/.../iics_test/`)|
| Source_File_Name | file name |
| File_Type | file type |
| File_Format | file format|
| File_Count | expected row count for that file |
| Is_Active | 1 = Active, 0 = Inactive|

The full source path used for reading is `Source_Path + Source_File_Name` + (add File_Format if needed).


## Required Libraries

In [0]:
from datetime import datetime
from pyspark.sql.functions import col, row_number, desc, current_user, lit
# from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType, DoubleType
from pyspark.sql import Row
import traceback
import uuid

## Configuration


In [0]:
catalog        = "iics"
schema         = "casemix"
control_table  = f"{catalog}.{schema}.csv_file_control_test"
log_table      = f"{catalog}.{schema}.csv_file_logs_test"

# Optional
force_reload   = False # set True to reprocess every file regardless of status
datamart_filter = "healthcare_test" # for parallel executions

print(f"Control table  : {control_table}")
print(f"Log table      : {log_table}")
print(f"Force reload   : {force_reload}")
print(f"DataMart filter: {datamart_filter}")

In [0]:
executed_by = spark.sql("SELECT current_user()").first()[0]

In [0]:
# spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

## Log Table Configs

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {log_table} (
  log_id          STRING,
  batch_id        STRING,
  datamart       STRING,
  file_name       STRING,
  source_path     STRING,
  tgt_catalog     STRING,
  tgt_schema      STRING,
  tgt_table       STRING,
  row_count       BIGINT,
  expected_count  BIGINT,
  status          STRING,
  error_message   STRING,
  executed_by     STRING,
  start_time      TIMESTAMP,
  end_time        TIMESTAMP,
  loaded_at       TIMESTAMP,
  duration_sec    DOUBLE
) USING DELTA
""")

print("Log Table Created")

In [0]:
log_schema = StructType([
    StructField("log_id",         StringType(),    True),
    StructField("batch_id",       StringType(),    True),
    StructField("datamart",       StringType(),    True),
    StructField("file_name",      StringType(),    True),
    StructField("source_path",    StringType(),    True),
    StructField("tgt_catalog",    StringType(),    True),
    StructField("tgt_schema",     StringType(),    True),
    StructField("tgt_table",      StringType(),    True),
    StructField("row_count",      LongType(),      True),
    StructField("expected_count", LongType(),      True),
    StructField("status",         StringType(),    True),
    StructField("error_message",  StringType(),    True),
    StructField("executed_by",    StringType(),    True),
    StructField("start_time",     TimestampType(), True),
    StructField("end_time",       TimestampType(), True),
    StructField("loaded_at",      TimestampType(), True),
    StructField("duration_sec",   DoubleType(),    True),
    # hash needs to be added
])